In [18]:
import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader
from dotenv import load_dotenv
import os

In [19]:
# Load Data
df_movies = pd.read_csv("../data/movies.csv")
df_ratings = pd.read_csv("../data/ratings.csv")
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [20]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    df_ratings[['userId', 'movieId', 'rating']], 
    reader
)
trainset = data.build_full_trainset()
svd_model = SVD()
svd_model.fit(trainset)

# Test prediction
prediction = svd_model.predict(uid=1, iid=100)
print(f"Predicted rating: {prediction.est}")

Predicted rating: 3.856146827346623


In [1]:
def get_cf_recommendations(user_id, n=5, genre=None):
    # Step 1 — watched movies
    watched = df_ratings[df_ratings['userId'] == user_id]['movieId'].tolist()
    
    # Step 2 — candidate pool
    all_movies = df_movies['movieId'].tolist()
    
    #  narrow the pool to the requested genre
    if genre:
        genre_ids = set(
            df_movies[df_movies["genres"].str.contains(genre, case=False)]["movieId"]
        )
        all_movies = [m for m in all_movies if m in genre_ids]
    
    unwatched = [m for m in all_movies if m not in watched]
    
    # Step 3 — predict ratings
    predictions = []
    for movie_id in unwatched:
        pred = svd_model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))
    
    # Step 4 — sort by predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Step 5 — get top n titles
    titles = []
    for movie_id, score in predictions[:n]:
        title = df_movies[df_movies['movieId'] == movie_id]['title'].values
        if len(title) > 0:
            titles.append(title[0])
    
    return titles

In [22]:
print(get_cf_recommendations(1, n=5))

['Shawshank Redemption, The (1994)', 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)', 'Philadelphia Story, The (1940)', 'North by Northwest (1959)', 'Casablanca (1942)']


In [23]:
#Check for differnt users
for uid in [1, 5, 50, 100]:
    print(f"\nUser {uid}:")
    recs = get_cf_recommendations(uid, n=3)
    for r in recs:
        print(f"  → {r}")


User 1:
  → Shawshank Redemption, The (1994)
  → Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
  → Philadelphia Story, The (1940)

User 5:
  → Spirited Away (Sen to Chihiro no kamikakushi) (2001)
  → Lawrence of Arabia (1962)
  → Forrest Gump (1994)

User 50:
  → Schindler's List (1993)
  → One Flew Over the Cuckoo's Nest (1975)
  → Laputa: Castle in the Sky (Tenkû no shiro Rapyuta) (1986)

User 100:
  → Bridge on the River Kwai, The (1957)
  → Streetcar Named Desire, A (1951)
  → Boondock Saints, The (2000)
